# MRRpy-Nepal — official flood methods next to the physics model

Run the next cell: it opens the **MRRpy-Nepal form**.

| Step | What you do |
|---|---|
| **1 Site** | a results folder of an earlier MRRpy run, **or** a DEM file and an outlet (delineated here), **or** the catchment numbers typed in — then **Measure the catchment** |
| **2 Methods** | choose **the manual** you design to (its methods are ticked) and **what you are designing** (its return periods are ticked); each ticked method opens only the inputs it needs |
| **3 Results** | **Compute the estimates**: the table with a validity flag per method, the official range, the physics model's design flood, a chart; save the table, chart and report |
| **Monthly and low flows** | MIP 1982, WECS/DHM and MHSP monthly flows, the low-flow rule and the e-flow |

The methods: WECS/DHM 1990 (HYDEST), DHM 2004 (Sharma & Adhikari), MHSP 1997, Modified Dickens,
the specific-discharge method (WECS 2019), the Rational method, frequency analysis at a gauge,
the catchment area ratio, and — for comparison only — the Indian formulas of Dickens, Ryves and
Inglis. Every coefficient comes from the compendium *Nepal Flood Estimation Methods*.

**Install:** `pip install "MRRpy[notebook]"` (ipywidgets).

In [ ]:
from MRRpy.nepal import NepalForm

form = NepalForm()
form

## The same in code — typed catchment numbers

No DEM at hand? Type the numbers. Here: the compendium's 65 km² snow-free Chure catchment
(longest flow path 15 km, slope 0.015), every method, every return period of the DoWRI pilot ToR.

In [ ]:
from MRRpy.nepal import NepalStudy

study = NepalStudy.from_numbers(
    65.0, flow_length_km=15.0, flow_path_slope=0.015, name="Chure example",
    agency="all", structure="study",
    inputs={"mhsp_region": "western", "runoff_coefficient": 0.45,
            "rain_24h": {2: 120, 100: 250}},
)
result = study.run()
result.table.drop(columns=["key", "notes"])

In [ ]:
result.envelope            # the range of the official regional methods

In [ ]:
for w in result.warnings():
    print("-", w)
print()
for line in result.recommendation:     # the compendium's quick guide for this catchment
    print("-", line)

In [ ]:
fig, ax = result.plot()

## With a DEM — the physics model's design flood

With a DEM, MRRpy-Nepal measures A, the area below 3,000 m and 5,000 m, the snow area, the flow
length and slope itself, and runs the MRRpy physics model with a design storm: the T-year 24-hour
rainfall spread with the Mononobe curve (the rule the Rational method uses). Uncomment and put in
your DEM and outlet:

In [ ]:
# study = NepalStudy.from_dem(
#     "dem.tif", (28.70, 80.95), "results",          # DEM, outlet (lat, lon), results folder
#     name="My khola", agency="dowri_2019", structure="weir",
#     methods=["wecs_dhm", "dhm_2004", "modified_dickens", "rational", "physics"],
#     inputs={"runoff_coefficient": 0.45, "rain_24h": {2: 120, 50: 230, 100: 250}},
#     physics={"curve_number": 75},
# )
# result = study.run()                 # delineate → measure → methods → physics → results/nepal/
# for T, flood in result.physics.items():
#     print(flood.summary())
# result.estimates.compare("physics")  # inside / above / below the official range

## Monthly and low flows

MIP 1982 from one dry-season measurement (Region 4, 1.5 m³/s on 15 February — the compendium's
example), next to the WECS/DHM and MHSP monthly regressions for a mean monsoon precipitation of
1,500 mm, with the "lower of WECS/DHM and MIP" low-flow rule and the e-flow release.

In [ ]:
from MRRpy.nepal import CatchmentDescriptors, monthly_flows

mf = monthly_flows(CatchmentDescriptors(area_km2=65.0), mip_region=4,
                   measurements=[("2024-02-15", 1.5)], mmp_mm=1500.0)
print("\n".join(mf.notes))
mf.table.round(2)

## Frequency analysis of a gauge or rain-gauge record

Gumbel, Log-Pearson III and GEV (L-moments), each with a 90 % bootstrap band — for annual maximum
floods at a gauge, or annual maximum daily rainfall at a DHM station (then use the fitted values as
`rain_24h`).

In [ ]:
from MRRpy.nepal import frequency

annual_max_rain = [112, 98, 140, 131, 176, 120, 158, 95, 149, 166,
                   127, 108, 205, 143, 119, 151, 104, 138, 181, 133]
frequency.compare_distributions(annual_max_rain, [2, 10, 50, 100])

More: the [MRRpy-Nepal pages](https://mrrpy.readthedocs.io/en/latest/nepal/) of the
documentation, `MRRpy nepal --help` in a terminal, and `NEPAL_PLAN.md` for what is included,
deferred or left out.